# **Notebook 6: Grad-CAM Interpretability for Bio_ClinicalBERT Residual Gated Fusion**

**Description:** This notebook evaluates the retained Bio_ClinicalBERT residual gated fusion checkpoint qualitatively instead of training another classifier. It rebuilds the gated model architecture, loads the original full-image checkpoint selected in Notebook 04 and retained after Notebook 05 tuning, then generates frontal-branch Grad-CAM heatmaps with prediction metadata.

**Main Questions This Notebook Should Answer:**
- Does the final selected model's visual evidence look plausible at the case level?
- How do correct and incorrect predictions differ in Grad-CAM activation?
- How should the heatmaps support the final case analysis without being treated as proof of clinical localization?

---

# **Part 1: Configuration**
### **Imports and Configuration**


In [ ]:
from pathlib import Path
import sys
import gc
import json
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms

from transformers import AutoModel, AutoTokenizer

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

project_root = Path.cwd().resolve()
if not (project_root / "splits").exists():
    project_root = project_root.parent

base_dir = project_root
split_dir = base_dir / "splits"
train_csv = split_dir / "train.csv"
val_csv = split_dir / "val.csv"
test_csv = split_dir / "test.csv"

HF_CACHE_DIR = base_dir / "hf_cache"
IMG_SIZE = 224
TEST_BATCH_SIZE = 1
NUM_WORKERS = 0
MAX_LEN = 64
TEXT_ENCODER_NAME = "emilyalsentzer/Bio_ClinicalBERT"
TOKENIZER_NAME = TEXT_ENCODER_NAME
TEXT_ENCODER_SPEC = {
    "name": "Bio_ClinicalBERT",
    "slug": "bio_clinicalbert",
    "loader": "auto",
    "model_name": TEXT_ENCODER_NAME,
    "tokenizer_name": TOKENIZER_NAME,
}

CHECKPOINT_PATH = (
    base_dir
    / "Experiments"
    / "outputs"
    / "experiment_04_segmentation"
    / "a_original_full"
    / "best_checkpoint.pt"
)

EXPERIMENT_NAME = "experiment_06_gradcam_interpretability"
OUTPUT_DIR = base_dir / "Experiments" / "outputs" / EXPERIMENT_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
HEATMAP_DIR = OUTPUT_DIR / "heatmaps"
HEATMAP_DIR.mkdir(parents=True, exist_ok=True)

print("Checkpoint:", CHECKPOINT_PATH)
print("Outputs:", OUTPUT_DIR)
print("Text encoder:", TEXT_ENCODER_NAME)
print("Fusion architecture: Bio_ClinicalBERT residual gated fusion")
print("Image pipeline: A_original_full from Notebook 04")

# Ensure Python can find the local xray_fusion package.
PROJECT_ROOT = next(
    (candidate for candidate in (Path.cwd(), *Path.cwd().parents) if (candidate / "src" / "xray_fusion").exists()),
    Path.cwd(),
)
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))

from xray_fusion.config import DEFAULT_IMAGE_SIZE, DEFAULT_MAX_LEN, find_project_root
from xray_fusion import data as xray_data
from xray_fusion import models as xray_models
from xray_fusion import training as xray_training
from xray_fusion import evaluation as xray_eval
from xray_fusion import segmentation as xray_segmentation
from xray_fusion import gradcam as xray_gradcam


### **Load splits, labels, text resources, and test dataset**


In [ ]:
class XRayMultimodalDataset(xray_data.XRayMultimodalDataset):
    def __init__(self, df, tokenizer, image_transform=None, max_len=MAX_LEN):
        super().__init__(df, tokenizer=tokenizer, image_transform=image_transform, max_len=max_len, label_key="labels")

    def __getitem__(self, idx):
        item = super().__getitem__(idx)
        row = self.df.iloc[idx]
        item["uid"] = str(row["uid"])
        item["clean_label"] = str(row["clean_label"])
        item["frontal_path"] = str(row["frontal_path"])
        item["lateral_path"] = str(row["lateral_path"])
        return item


checkpoint = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
checkpoint_encoder_spec = checkpoint.get("encoder_spec", TEXT_ENCODER_SPEC)
if checkpoint_encoder_spec.get("model_name") != TEXT_ENCODER_NAME:
    raise ValueError(
        "Notebook 06 expected the Bio_ClinicalBERT checkpoint, but the checkpoint metadata says "
        f"{checkpoint_encoder_spec.get('model_name')!r}."
    )
if checkpoint.get("fusion_block") != "ResidualGatedFusionBlock":
    raise ValueError(
        "Notebook 06 expected a residual gated fusion checkpoint from Notebook 03/04."
    )

train_df = pd.read_csv(train_csv)
val_df = pd.read_csv(val_csv)
test_df = pd.read_csv(test_csv)

label_names = list(checkpoint.get("label_names", sorted(train_df["clean_label"].unique())))
label_to_idx = {label: idx for idx, label in enumerate(label_names)}
for df in [train_df, val_df, test_df]:
    df["label_enc"] = df["clean_label"].map(label_to_idx)
    if df["label_enc"].isna().any():
        missing = sorted(df.loc[df["label_enc"].isna(), "clean_label"].unique())
        raise ValueError(f"Found labels not present in checkpoint label_names: {missing}")
    df["label_enc"] = df["label_enc"].astype(int)

num_classes = len(label_names)

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME, cache_dir=HF_CACHE_DIR)
image_transform = transforms.Compose(
    [
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
    ]
)

test_dataset = XRayMultimodalDataset(test_df, tokenizer=tokenizer, image_transform=image_transform)
test_loader = DataLoader(
    test_dataset,
    batch_size=TEST_BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
)

print("Loaded checkpoint:", CHECKPOINT_PATH)
print("Checkpoint architecture:", checkpoint.get("model_architecture"))
print("Checkpoint text encoder:", checkpoint.get("text_encoder"))
print("Checkpoint image pipeline:", checkpoint.get("pipeline_spec", {}).get("slug"))
print("Test rows:", len(test_df))
print("Classes:", label_names)


---

# **Part 2: Model and Grad-CAM Utilities**

### **Rebuild Bio_ClinicalBERT residual gated fusion model and load checkpoint**


In [ ]:
DualImageEncoder = xray_models.DualImageEncoder
ResidualGatedFusionBlock = xray_models.ResidualGatedFusionBlock


class BioClinicalBertGatedFusionClassifier(xray_models.SelectedEncoderGatedFusionModel):
    def __init__(self, num_classes, freeze_bert=True, cache_dir=None):
        super().__init__(
            encoder_spec=checkpoint_encoder_spec,
            num_classes=num_classes,
            fusion_block=None,
            fusion_dim=256,
            fusion_dropout=0.2,
            freeze_bert=freeze_bert,
            freeze_image_backbone=False,
            cache_dir=cache_dir,
        )


model = BioClinicalBertGatedFusionClassifier(
    num_classes=num_classes,
    freeze_bert=bool(checkpoint.get("freeze_bert", True)),
    cache_dir=HF_CACHE_DIR,
).to(device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

target_conv = model.image_encoder.backbone.layer4[-1].conv2

print("Loaded Bio_ClinicalBERT residual gated fusion checkpoint.")
print("Target Grad-CAM layer:", target_conv)


### **Define Grad-CAM helpers**


In [ ]:
GradCAM = xray_gradcam.GradCAM
tensor_to_image = xray_gradcam.tensor_to_image
overlay_heatmap = xray_gradcam.overlay_heatmap
save_cam_panel = xray_gradcam.save_cam_panel


---

# **Part 3: Generate Interpretability Artifacts**

### **Run Grad-CAM on the test set and save outputs**


In [ ]:
grad_cam = GradCAM(model, target_conv)

records = []
all_true = []
all_pred = []

for batch in tqdm(test_loader, desc="Grad-CAM Test Sweep", dynamic_ncols=True):
    frontal = batch["frontal"].to(device)
    lateral = batch["lateral"].to(device)
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    labels = batch["labels"].to(device)

    cam, logits = grad_cam.generate(
        frontal=frontal,
        lateral=lateral,
        input_ids=input_ids,
        attention_mask=attention_mask,
    )

    pred_idx = int(logits.argmax(dim=1).item())
    true_idx = int(labels.item())
    confidence = float(torch.softmax(logits, dim=1)[0, pred_idx].item())
    correct = pred_idx == true_idx

    frontal_image = tensor_to_image(batch["frontal"][0])
    overlay = overlay_heatmap(frontal_image, cam)

    label_slug = batch["clean_label"][0].replace(" ", "_")
    uid = str(batch["uid"][0])
    sample_dir = HEATMAP_DIR / ("correct" if correct else "incorrect") / label_slug
    sample_dir.mkdir(parents=True, exist_ok=True)
    save_path = sample_dir / f"{uid}_gradcam.png"

    save_cam_panel(
        frontal_image,
        cam,
        overlay,
        save_path,
        f"Pred: {label_names[pred_idx]} | True: {label_names[true_idx]}",
    )

    records.append(
        {
            "uid": uid,
            "true_label": label_names[true_idx],
            "pred_label": label_names[pred_idx],
            "correct": correct,
            "confidence": confidence,
            "cam_mean": float(cam.mean()),
            "cam_std": float(cam.std()),
            "cam_max": float(cam.max()),
            "cam_peak_row": int(np.unravel_index(np.argmax(cam), cam.shape)[0]),
            "cam_peak_col": int(np.unravel_index(np.argmax(cam), cam.shape)[1]),
            "frontal_path": batch["frontal_path"][0],
            "lateral_path": batch["lateral_path"][0],
            "heatmap_path": str(save_path),
        }
    )

    all_true.append(true_idx)
    all_pred.append(pred_idx)

grad_cam.close()

gradcam_df = pd.DataFrame(records)
gradcam_df.to_csv(OUTPUT_DIR / "gradcam_summary.csv", index=False)

cm = confusion_matrix(all_true, all_pred)
report = classification_report(
    all_true,
    all_pred,
    target_names=label_names,
    output_dict=True,
    zero_division=0,
)

pd.DataFrame(cm, index=label_names, columns=label_names).to_csv(
    OUTPUT_DIR / "test_confusion_matrix.csv"
)
with open(OUTPUT_DIR / "test_classification_report.json", "w", encoding="utf-8") as handle:
    json.dump(report, handle, indent=2)

print("Saved:", OUTPUT_DIR / "gradcam_summary.csv")
display(gradcam_df.head())


---

# **Part 4: Results and Reporting**

### **Summarize correct vs incorrect attention patterns**


In [ ]:
summary_by_correctness = (
    gradcam_df.groupby("correct")[["cam_mean", "cam_std", "cam_max", "confidence"]]
    .mean()
    .reset_index()
)
summary_by_class = (
    gradcam_df.groupby(["true_label", "correct"])[["cam_mean", "cam_std", "cam_max", "confidence"]]
    .mean()
    .reset_index()
)

summary_by_correctness.to_csv(OUTPUT_DIR / "attention_summary_by_correctness.csv", index=False)
summary_by_class.to_csv(OUTPUT_DIR / "attention_summary_by_class.csv", index=False)

display(summary_by_correctness)
display(summary_by_class)


### **Show representative examples for the report**


In [ ]:
correct_examples = (
    gradcam_df[gradcam_df["correct"]]
    .sort_values("confidence", ascending=False)
    .groupby("true_label")
    .head(1)
)
incorrect_examples = (
    gradcam_df[~gradcam_df["correct"]]
    .sort_values("confidence", ascending=False)
    .groupby("true_label")
    .head(1)
)

display(correct_examples[["uid", "true_label", "pred_label", "confidence", "heatmap_path"]])
display(incorrect_examples[["uid", "true_label", "pred_label", "confidence", "heatmap_path"]])


---

# **Part 5: Conclusion**

This notebook explains the retained Bio_ClinicalBERT residual gated fusion checkpoint using the original full frontal/lateral image pipeline selected after Notebook 04. This placement matters because Grad-CAM should explain the final selected image pipeline and checkpoint, not a segmentation-preprocessed variant that the ablation rejected or a tuned model that did not outperform the original full-image checkpoint. The Grad-CAM sweep reproduces the same held-out test profile as the selected checkpoint. That confirms the interpretability artifacts are tied to the correct gated-fusion model.

The heatmaps provide useful qualitative examples, but they also expose an important limitation. Correct predictions are more confident on average (0.964 vs 0.840 for incorrect predictions), yet the CAM signal is often weak or absent: about 65% of generated maps have zero activation after the Grad-CAM ReLU step. This is especially common for confident normal predictions, so a blank or diffuse map should not be interpreted as strong clinical localization.

The most useful Grad-CAM cases are the abnormal examples with nonzero frontal activation. Correct cardiomegaly, chronic lung disease, pleural effusion, and pneumonia/opacity cases show stronger average CAM intensity than many incorrect cases, suggesting that the frontal image branch can contribute meaningful localized evidence when the model identifies an abnormal finding. However, many high-confidence errors are abnormal studies predicted as normal, often with near-zero CAM activation, which supports the earlier metric finding that minority-class recall remains the main weakness.

The Notebook 06 results overall support keeping the Bio_ClinicalBERT residual gated fusion model as the final reported model, but the Grad-CAM evidence should be treated as a failure analysis instead of proof of reliable localization. The saved correct and incorrect heatmap panels are most useful for the report as representative examples of when the model appears visually grounded and when it overconfidently misses abnormal findings.

---